# ADVS — Siamese CNN Signature Verifier

Model 3 of 4 — ResNet-50 twin → 128-D L2-normalised embedding → L1 distance → sigmoid match probability (`training_script.md` §3). Trains on **CEDAR** (55 signers × 24 genuine + 24 skilled forgeries; §4 below downloads and reshapes it), using real forgeries for the forged pairs and synthetic forgeries only as a fallback. Saves `siamese_signature.h5`, `siamese_encoder.h5`, and an EER `signature_threshold.txt`.

Preprocessing is **RGB + `resnet50.preprocess_input`** — identical to the ADVS API's serve path (`api/routers/signature.py`), so the embeddings and EER threshold transfer 1:1.

> Self-contained notebook. The code cell below is the full source of `scripts/train_signature.py` (definitions only); the run cell at the bottom launches training. Edit `DATA_ROOT` to point at your data.

## 1. Install dependencies (Colab / fresh env)

**On Colab this is REQUIRED, not optional** — Colab ships a newer TF/Keras whose saved `.h5` files the local ADVS venv (pinned to TF 2.16) may refuse to load; the pin keeps both sides on the same TF/Keras majors so the artefacts transfer. Run the cell, then **Runtime ▸ Restart session**, then continue from §2. Skip only if your environment already runs TF 2.16.

In [ ]:
%pip install -q "tensorflow==2.16.*" scikit-learn opencv-python-headless pillow
# Colab: now Runtime > Restart session, then continue from section 2.

## 2. Training code (from `scripts/train_signature.py`)

Running this cell only defines functions — it does not start training.

In [ ]:
"""ADVS - Siamese CNN signature verifier (model 3 of 4).

Verifies whether two signatures belong to the same vendor. ResNet-50 backbone ->
128-D L2-normalised embedding (twin) -> L1 distance -> sigmoid match probability.
Images are RGB and go through ``resnet50.preprocess_input`` — the SAME
preprocessing the ADVS API uses at serve time (api/routers/signature.py), so
embeddings and the EER threshold transfer 1:1. Forged pairs use REAL skilled
forgeries when a vendor ships them (e.g. CEDAR: ``<vendor>/forged/*.png``);
otherwise synthetic forgeries are fabricated from genuine samples (elastic +
rotation + noise). Faithful to training_script.md §3.

Data layout (read-only):
    <data-root>/training/signature_data/<vendor>/*.png           (genuine signatures)
    <data-root>/training/signature_data/<vendor>/forged/*.png    (optional real forgeries)
    <data-root>/validation/signature_data/<vendor>/*.png         (same shape)

Outputs (under <models-out>):
    siamese_signature.h5, siamese_encoder.h5, signature_threshold.txt (EER)

Run:
    python scripts/train_signature.py                 # full training (needs ML stack + data)
    python scripts/train_signature.py --dry-run       # validate layout only (stdlib only)
    python scripts/train_signature.py --smoke         # 1-epoch tiny CPU run (needs ML stack)

Heavy imports (tensorflow, cv2) are lazy so --dry-run works with only stdlib.
"""

from __future__ import annotations

import argparse
import os
import sys
import time
from pathlib import Path

try:
    PY_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    PY_ROOT = Path.cwd()  # notebook fallback - assumes cwd is python/

CONFIG: dict = {
    "image_size": 224,
    "embedding_dim": 128,
    "batch_size": 16,
    "epochs": 20,
    "lr": 1e-4,
    "pairs_per_vendor": 20,    # genuine+forged pairs generated per vendor
    "seed": 42,
}
SMOKE_OVERRIDES = {
    "image_size": 64,
    "batch_size": 2,
    "epochs": 1,
    "pairs_per_vendor": 4,
}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def log(msg: str) -> None:
    print(f"[signature] {msg}", flush=True)


def section(title: str) -> None:
    print("\n" + "=" * 70 + f"\n  {title}\n" + "=" * 70, flush=True)


class TrainError(RuntimeError):
    """Expected, user-actionable failure (e.g. missing data)."""


def require_dir(path: Path, what: str) -> None:
    if not path.is_dir():
        raise TrainError(f"Missing {what}: expected directory '{path}'.")


def vendor_dirs(path: Path) -> list[str]:
    return sorted(d.name for d in path.iterdir() if d.is_dir())


def validate_structure(train_dir: Path, val_dir: Path) -> dict:
    require_dir(train_dir, "signature training vendors")
    require_dir(val_dir, "signature validation vendors")
    tv = vendor_dirs(train_dir)
    vv = vendor_dirs(val_dir)
    if not tv:
        raise TrainError(f"No vendor subfolders under {train_dir}.")
    with_real_forgeries = sum(1 for v in tv if (train_dir / v / "forged").is_dir())
    return {
        "train_vendors": len(tv),
        "val_vendors": len(vv),
        "train_vendors_with_real_forgeries": with_real_forgeries,
    }


def make_synthetic_forgery(img, seed: int):
    """Elastic deformation (cv2.remap) + rotation + gaussian noise -> forgery."""
    import cv2
    import numpy as np

    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    dx = cv2.GaussianBlur(rng.uniform(-1, 1, (h, w)).astype("float32"), (0, 0), sigmaX=4) * 6
    dy = cv2.GaussianBlur(rng.uniform(-1, 1, (h, w)).astype("float32"), (0, 0), sigmaX=4) * 6
    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    warped = cv2.remap(img, (xx + dx).astype("float32"), (yy + dy).astype("float32"),
                       interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    angle = float(rng.uniform(-12, 12))
    rot = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    warped = cv2.warpAffine(warped, rot, (w, h), borderMode=cv2.BORDER_REFLECT)
    noise = rng.normal(0, 12, warped.shape).astype("float32")
    return np.clip(warped.astype("float32") + noise, 0, 255).astype("uint8")


def load_images(folder: Path, size: int) -> list:
    """RGB uint8 arrays, resized — RGB to match the API's PIL serve path."""
    import cv2

    imgs = []
    if not folder.is_dir():
        return imgs
    for ip in sorted(folder.iterdir()):
        if ip.suffix.lower() not in IMG_EXTS:
            continue
        arr = cv2.imread(str(ip))
        if arr is not None:
            imgs.append(cv2.resize(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB), (size, size)))
    return imgs


def load_vendors(root: Path, size: int) -> dict:
    """vendor -> {"genuine": [...], "forged": [...]} (forged = real skilled
    forgeries from the optional ``<vendor>/forged/`` subfolder, may be empty)."""
    out: dict[str, dict] = {}
    for vdir in sorted(d for d in root.iterdir() if d.is_dir()):
        genuine = load_images(vdir, size)
        if len(genuine) >= 2:
            out[vdir.name] = {"genuine": genuine, "forged": load_images(vdir / "forged", size)}
    return out


def build_encoder(cfg: dict):
    from tensorflow.keras import layers, models
    from tensorflow.keras.applications import ResNet50

    size = cfg["image_size"]
    backbone = ResNet50(weights="imagenet", include_top=False, pooling="avg",
                        input_shape=(size, size, 3))
    inp = layers.Input(shape=(size, size, 3))
    x = layers.Dense(cfg["embedding_dim"], activation=None)(backbone(inp))
    # UnitNormalization == L2-normalise. A real layer, NOT a Lambda: Keras 3
    # cannot pickle a lambda that closes over the lazily-imported tf module,
    # and the API's plain load_model() refuses Lambdas under default safe_mode.
    x = layers.UnitNormalization(axis=-1, name="l2norm")(x)
    return models.Model(inp, x, name="siamese_encoder")


def l1_distance(tensors):
    """Module-level named function (no closure) so Keras can serialise the
    twin model's Lambda layer into siamese_signature.h5."""
    import tensorflow as tf

    return tf.math.abs(tensors[0] - tensors[1])


def equal_error_rate_threshold(distances, labels) -> float:
    """Distance threshold where FAR ~= FRR. label 1 = genuine (small distance)."""
    import numpy as np

    labels = np.asarray(labels)
    lo, hi = float(distances.min()), float(distances.max())
    if hi <= lo:
        return float(lo)
    best_t, best_gap = lo, 1e9
    for t in np.linspace(lo, hi, 200):
        pred_genuine = distances <= t
        far = float(np.mean(pred_genuine[labels == 0])) if np.any(labels == 0) else 0.0
        frr = float(np.mean(~pred_genuine[labels == 1])) if np.any(labels == 1) else 0.0
        gap = abs(far - frr)
        if gap < best_gap:
            best_gap, best_t = gap, float(t)
    return best_t


def train(cfg: dict, train_dir: Path, val_dir: Path, models_out: Path) -> None:
    section("Siamese CNN signature verification")
    import numpy as np
    import tensorflow as tf
    from tensorflow.keras import layers, models
    from tensorflow.keras.applications.resnet50 import preprocess_input

    size = cfg["image_size"]
    rng = np.random.default_rng(cfg["seed"])

    train_vendors = load_vendors(train_dir, size)
    val_vendors = load_vendors(val_dir, size)
    if len(train_vendors) < 1:
        raise TrainError("Need >= 1 training vendor with >= 2 genuine signatures.")
    n_real = sum(1 for d in train_vendors.values() if d["forged"])
    log(f"vendors -> {len(train_vendors)} train ({n_real} with real forgeries) / {len(val_vendors)} val")

    def make_pairs(vendors: dict, n_per_vendor: int):
        pa, pb, labels = [], [], []
        for data in vendors.values():
            imgs = data["genuine"]
            real_forged = data["forged"]
            for k in range(n_per_vendor):
                if k % 2 == 0:  # genuine pair
                    i, j = rng.choice(len(imgs), size=2, replace=len(imgs) < 2)
                    pa.append(imgs[i]); pb.append(imgs[j]); labels.append(1)
                else:           # forged pair — real skilled forgery when available
                    i = int(rng.integers(len(imgs)))
                    if real_forged:
                        forged = real_forged[int(rng.integers(len(real_forged)))]
                    else:
                        forged = make_synthetic_forgery(imgs[i], seed=int(rng.integers(1 << 30)))
                    pa.append(imgs[i]); pb.append(forged); labels.append(0)
        # resnet50.preprocess_input == the API's serve-time preprocessing
        # (api/routers/signature.py) — keep them identical or the EER
        # threshold does not transfer.
        a = preprocess_input(np.asarray(pa, dtype="float32"))
        b = preprocess_input(np.asarray(pb, dtype="float32"))
        return a, b, np.asarray(labels, dtype="float32")

    encoder = build_encoder(cfg)
    in_a = layers.Input(shape=(size, size, 3))
    in_b = layers.Input(shape=(size, size, 3))
    distance = layers.Lambda(l1_distance, name="l1_distance",
                             output_shape=(cfg["embedding_dim"],))(
        [encoder(in_a), encoder(in_b)]
    )
    out = layers.Dense(1, activation="sigmoid")(distance)
    siamese = models.Model([in_a, in_b], out, name="siamese_signature")
    siamese.compile(optimizer=tf.keras.optimizers.Adam(cfg["lr"]),
                    loss="binary_crossentropy", metrics=["accuracy"])

    train_a, train_b, train_y = make_pairs(train_vendors, cfg["pairs_per_vendor"])
    fit_kwargs = {"epochs": cfg["epochs"], "batch_size": cfg["batch_size"]}
    if val_vendors:
        val_a, val_b, val_y = make_pairs(val_vendors, cfg["pairs_per_vendor"])
        fit_kwargs["validation_data"] = ([val_a, val_b], val_y)
    else:
        val_a = val_b = val_y = None
    log(f"training pairs: {len(train_y)}")

    siamese.fit([train_a, train_b], train_y, **fit_kwargs)

    siamese.save(str(models_out / "siamese_signature.h5"))
    encoder.save(str(models_out / "siamese_encoder.h5"))

    # EER threshold on validation (fall back to training pairs if no val vendors).
    ea_src, eb_src, y_src = (val_a, val_b, val_y) if val_vendors else (train_a, train_b, train_y)
    dists = np.linalg.norm(encoder.predict(ea_src, verbose=0) - encoder.predict(eb_src, verbose=0), axis=1)
    threshold = equal_error_rate_threshold(dists, y_src)
    (models_out / "signature_threshold.txt").write_text(f"{threshold:.6f}\n")
    log(f"Saved siamese_signature.h5, siamese_encoder.h5, signature_threshold.txt (EER={threshold:.4f})")

    section("Inference sanity check")
    probe = preprocess_input(np.random.rand(1, size, size, 3).astype("float32") * 255.0)
    emb = encoder.predict(probe, verbose=0)
    log(f"encoder -> embedding dim {emb.shape[1]}")


def parse_args(argv: list[str]) -> argparse.Namespace:
    ap = argparse.ArgumentParser(description="Train the ADVS Siamese signature verifier.")
    ap.add_argument("--data-root", default=str(PY_ROOT / "data"))
    ap.add_argument("--models-out", default=str(PY_ROOT / "models"))
    ap.add_argument("--dry-run", action="store_true",
                    help="Validate layout/config only; no heavy imports, no training.")
    ap.add_argument("--smoke", action="store_true",
                    help="Tiny 1-epoch CPU run (needs ML stack + fixtures).")
    return ap.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv if argv is not None else sys.argv[1:])
    cfg = dict(CONFIG)
    if args.smoke:
        cfg.update(SMOKE_OVERRIDES)
        os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
        log("SMOKE MODE: reduced epochs/sizes, CPU only.")

    data_root = Path(args.data_root)
    train_dir = data_root / "training" / "signature_data"
    val_dir = data_root / "validation" / "signature_data"
    models_out = Path(args.models_out)
    log(f"data-root={data_root}  models-out={models_out}")

    try:
        summary = validate_structure(train_dir, val_dir)
    except TrainError as exc:
        log(f"STRUCTURE ERROR: {exc}")
        return 2
    log(f"ok: {summary}")

    if args.dry_run:
        section("DRY RUN - structure valid, skipping training")
        return 0

    models_out.mkdir(parents=True, exist_ok=True)
    started = time.time()
    train(cfg, train_dir, val_dir, models_out)
    section(f"Done in {time.time() - started:.1f}s - artefacts in {models_out}")
    return 0


## 3. Configure paths

`DATA_ROOT` must contain `training/` and `validation/` subfolders for this model (see `python/data/README.md`).

In [ ]:
from pathlib import Path
DATA_ROOT = str(PY_ROOT / "data")     # edit to your dataset location
MODELS_OUT = str(PY_ROOT / "models")
print("data:", DATA_ROOT, "
out :", MODELS_OUT)

## 4. Get CEDAR and reshape it into the ADVS layout

Downloads the CEDAR signature dataset (55 signers × 24 genuine + 24 skilled forgeries, research use) and rebuilds `DATA_ROOT`'s `signature_data/` as `signer_XXX/*.png` (genuine) + `signer_XXX/forged/*.png` (real forgeries), holding out the last 11 signers (~20%) for validation.

> **This replaces any existing `signature_data/` under `DATA_ROOT`** (locally that's just the regenerable smoke fixtures). Skip this cell if you already have real per-vendor signature data in place.

In [ ]:
# CEDAR: 55 signers x 24 genuine + 24 skilled forgeries (research use).
import re
import shutil
import subprocess
import urllib.request
from pathlib import Path

CEDAR_URL = "http://www.cedar.buffalo.edu/NIJ/data/signatures.rar"
VAL_SIGNERS = 11                      # 11 of 55 signers (~20%) -> validation
work = Path("cedar_raw")
work.mkdir(exist_ok=True)
rar = work / "signatures.rar"

if not list(work.rglob("full_org")):
    if not rar.exists():
        print("downloading", CEDAR_URL)
        urllib.request.urlretrieve(CEDAR_URL, rar)
    subprocess.run(["apt-get", "-qq", "install", "-y", "unrar"], check=False)
    subprocess.run(["unrar", "x", "-o+", str(rar), str(work)], check=True)
# If the buffalo.edu mirror is down: upload kaggle.json, then
#   %pip -q install kaggle
#   !kaggle datasets download -d shreelakshmigp/cedardataset -p cedar_raw --unzip
# and re-run this cell (it finds full_org/full_forg wherever they land).

org = next(work.rglob("full_org"))    # original_<signer>_<n>.png
forg = next(work.rglob("full_forg"))  # forgeries_<signer>_<n>.png
signer_re = re.compile(r"_(\d+)_\d+\.png$", re.IGNORECASE)

data_root = Path(DATA_ROOT)
for split in ("training", "validation"):
    target = data_root / split / "signature_data"
    if target.exists():
        shutil.rmtree(target)         # replace fixtures / an old CEDAR copy

n_train = n_val = 0
for src, sub in ((org, ""), (forg, "forged")):
    for img in sorted(src.glob("*.png")):
        m = signer_re.search(img.name)
        if not m:
            continue
        signer = int(m.group(1))
        split = "validation" if signer > 55 - VAL_SIGNERS else "training"
        dest = data_root / split / "signature_data" / f"signer_{signer:03d}" / sub
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy2(img, dest / img.name)
        if not sub:
            n_train += split == "training"
            n_val += split == "validation"
print(f"CEDAR ready: {n_train} genuine train / {n_val} genuine val images"
      f" (+ skilled forgeries in each signer's forged/)")

## 5. Validate the data layout (no training)

A quick structural check before committing to a full run — the summary should show 44 train / 11 val signers, all with real forgeries.

In [ ]:
main(["--data-root", DATA_ROOT, "--models-out", MODELS_OUT, "--dry-run"])

## 6. Train

Full run (GPU recommended — Runtime ▸ Change runtime type ▸ GPU). Use `--smoke` instead for a fast 1-epoch CPU sanity check.

In [ ]:
main(["--data-root", DATA_ROOT, "--models-out", MODELS_OUT])
# Fast sanity run instead: main(["--data-root", DATA_ROOT, "--models-out", MODELS_OUT, "--smoke"])

## 7. Download the trained artefacts

Drop all three files into the repo's `python/models/` — the ADVS API picks them up on the next boot (`SIAMESE_MODEL_PATH` defaults to `siamese_encoder.h5`; the threshold is read from `signature_threshold.txt`).

In [ ]:
from pathlib import Path

artefacts = [Path(MODELS_OUT) / n for n in
             ("siamese_signature.h5", "siamese_encoder.h5", "signature_threshold.txt")]
try:
    from google.colab import files
    for p in artefacts:
        files.download(str(p))
except ImportError:
    print("Not on Colab - copy these into the repo's python/models/:")
    for p in artefacts:
        print(" ", p)